# 🐇 Light Rabbit — pannello del nodo White Rabbit su KR260

Carica il firmware **v17**, porta la scheda in **`TRACK_PHASE`** contro il WR-ZEN,
e mostra **vUART**, **frequenzimetro**, stato del lock e **TDC** (build_id, PPS,
calibrazione code-density).

### ⚠️ Prima di eseguire
| | |
|---|---|
| **Fibra** | **SFP BiDi verso il WR-ZEN** (non loopback) |
| **Kernel** | dev'essere **root** (serve `/dev/mem` + `xmutil`) — jupyter va lanciato con `sudo` |
| **vUART** | **un solo consumatore alla volta**: se hai uno script/monitor attivo da shell, chiudilo o le celle qui daranno output vuoto |
| **AXI** | non leggere `0xA00xxxxx` **prima** di aver caricato l'app (cella 2) → blocca la scheda |

Eseguire le celle **in ordine** la prima volta.

> **Riprendere una sessione** (kernel nuovo, firmware gia' caricato e nodo agganciato):
> esegui la **cella 1** e poi vai dove vuoi — le celle si agganciano all'AXI da sole.
> **NON rieseguire la cella 2**: ricaricherebbe la PL e butterebbe via il lock.

## 1 — Helper: mmap, vUART, frequenzimetro

In [ ]:
import mmap, os, ctypes, time, subprocess, re
from datetime import datetime

WB_BASE, WB_SPAN = 0xA0000000, 0x10000     # bridge Wishbone: vUART + WRPC
FM_BASE, FM_SPAN = 0xA0030000, 0x1000      # freq_counter (TXUSRCLK2)
TDR, RDR, MAGIC  = 0x510, 0x514, 0x000     # registri vUART / magic 'WRPC'
FM_GATE, FM_CNT, FM_STAT = 0x00, 0x04, 0x08

assert os.geteuid() == 0, "Il kernel NON gira come root: rilancia jupyter con sudo"

_mem = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)

def _map(base, span):
    return mmap.mmap(_mem, span, mmap.MAP_SHARED,
                     mmap.PROT_READ | mmap.PROT_WRITE, offset=base)

_wb = _fm = None                      # mappati DOPO il load (cella 2)

def rd(m, off):      return ctypes.c_uint32.from_buffer(m, off).value
def wr(m, off, val): ctypes.c_uint32.from_buffer(m, off).value = val & 0xffffffff

def attach():
    """Mappa i registri PL. Da chiamare SOLO dopo xmutil loadapp."""
    global _wb, _fm
    _wb, _fm = _map(WB_BASE, WB_SPAN), _map(FM_BASE, FM_SPAN)
    m = rd(_wb, MAGIC)
    if m != 0x57525043:               # 'WRPC'
        raise RuntimeError(f"MAGIC=0x{m:08X} != 'WRPC' → app WR non caricata")
    print(f"✅ bridge WB agganciato (MAGIC='WRPC') @ 0x{WB_BASE:08X}")

def ensure_attached():
    """Aggancia l'AXI se non lo e' gia'. Serve quando si riprende una sessione senza
    rieseguire la cella 2 (che ricaricherebbe la PL e butterebbe via il lock)."""
    if _wb is None:
        attach()

# ---------- vUART: parla con la console del softcore WRPC (LM32) ----------
def _drain(quiet=0.30, hard=3.0):
    out, t0, last = bytearray(), time.time(), time.time()
    while time.time() - t0 < hard:
        v = rd(_wb, RDR)
        if v & 0x100:                        # bit8 = byte valido
            out.append(v & 0xff); last = time.time()
        elif time.time() - last > quiet:
            break
        else:
            time.sleep(0.002)
    return out.decode('ascii', 'replace')

def wrc(cmd="", quiet=0.30, hard=3.0, echo=True):
    """Manda un comando alla vUART e restituisce l'output."""
    ensure_attached()
    _drain(0.05, 0.3)                        # svuota il pendente
    for ch in (cmd + "\r").encode('ascii', 'replace'):
        for _ in range(200000):
            if rd(_wb, TDR) & 0x100:         # TX ready
                break
        wr(_wb, TDR, ch)
    time.sleep(0.05)
    txt = _drain(quiet, hard)
    if echo:
        print(f"wrc# {cmd}\n{txt}")
    return txt

def stat():
    """'stat' è un TOGGLE dell'output continuo → ritenta finché trova la riga lnk:."""
    for _ in range(3):
        t = wrc("stat", echo=False)
        if "lnk:" in t:
            return t
        time.sleep(0.7)
    return t

def parse_stat(txt):
    """Estrae i campi k:v dell'ultima riga lnk: (la più fresca)."""
    lines = [l for l in txt.splitlines() if "lnk:" in l]
    if not lines:
        return {}
    d = dict(re.findall(r"(\w+):('[^']*'|-?\w+)", lines[-1]))
    return {k: v.strip("'") for k, v in d.items()}

# ---------- frequenzimetro ----------
def freq_meas(gate_cycles=125_000_000, pl_clk_hz=125_000_000):
    """Misura TXUSRCLK2. Ritorna (Hz, conteggio grezzo, secondi di gate)."""
    ensure_attached()
    wr(_fm, FM_GATE, gate_cycles)
    gate_s = gate_cycles / pl_clk_hz
    time.sleep(gate_s + 0.4)
    rd(_fm, FM_STAT)                          # la lettura azzera il flag
    time.sleep(gate_s + 0.4)
    cnt = rd(_fm, FM_CNT)
    return cnt / gate_s, cnt, gate_s

def sh(cmd, timeout=90):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return (r.stdout + r.stderr).strip()

print("helper pronti — NON è ancora stato toccato l'AXI (si mappa nella cella 2)")

## 2 — Carica il firmware v17 sulla PL

`xmutil unloadapp` → `loadapp` → verifica che i **timestamp di `/dev/uio*` siano freschi**
(è la prova che la PL è stata **programmata davvero**, non un deploy finto).

👀 **Guarda la scheda:** subito dopo il load **LED2** (cage SFP) deve **respirare** = fabric vivo.

Per tornare a una versione precedente, cambia solo `APP` nella cella sotto (es.
`"kr260_wr_sdm_app_v15"`) — le sezioni 10/11/12 (build_id, TDC) si rifiutano da sole di
girare su immagini troppo vecchie (< v16).

In [ ]:
APP = "kr260_wr_sdm_app_v17"

print(sh("xmutil unloadapp") or "(niente da scaricare)")
time.sleep(1)
print(sh(f"xmutil loadapp {APP}"))
time.sleep(3)

print("\n--- /dev/uio (uio4/5/6 = app WR; devono avere l'ora DI ADESSO) ---")
print(sh("for u in /sys/class/uio/uio*; do echo \"  $(basename $u): $(cat $u/name)\"; done"))
print(sh("ls -la --time-style=+%H:%M:%S /dev/uio* | awk '{print \"  \"$6\"  \"$7}'"))
print(f"\n  adesso sono le {datetime.now():%H:%M:%S}")

attach()          # ora è lecito toccare l'AXI

## 3 — Laser SFP + livelli ottici

⚠️ **Trappola:** ogni riprogrammazione della PL **azzera il GPO** che abilita il laser →
va riacceso **dopo ogni load**, altrimenti il TX resta spento e il link non sale mai.

Atteso: **RX ≈ −6 dBm** (se è LOS/garbage: fibra staccata o SFP rotto → fermarsi qui).

In [ ]:
print(sh("python3 /home/ubuntu/tx_enable.py on"))
print()
ddm = sh("python3 /home/ubuntu/read_sfp_ddm.py")
print("\n".join(l for l in ddm.splitlines()
                if re.search(r"power|Temperatura|bias|LOS|TX_DISABLE|fault", l, re.I)))

## 4 — vUART: la console del softcore WRPC

La `wrc()` parla direttamente col softcore LM32 dentro la PL, via il bridge Wishbone.
Se il softcore non fosse partito, qui non uscirebbe **nulla** (vUART muta = wrc non boota).

In [ ]:
time.sleep(6)                 # lascia bootare il wrc dopo il load
wrc("")                       # prompt: deve rispondere 'wrc#'
wrc("help", hard=4.0);

## 5 — Bring-up White Rabbit: `slave` + PTP

MAC fisso, **autoneg ON** (col partner reale deve completare), modo **slave**, `ptp start`.

In [ ]:
for c in ["ptp stop", "mac set 00:11:22:33:44:55", "mdio an 1", "mode slave", "ptp start"]:
    wrc(c); time.sleep(1)
print("→ bring-up lanciato; il lock di fase arriva tipicamente in pochi secondi")

## 6 — Frequenzimetro: la QPLL0 sta davvero girando alla frequenza giusta?

Conta i fronti di **TXUSRCLK2** (il clock che esce dalla QPLL0 sterzata via SDM) su una
finestra di gate. **Atteso ≈ 62.5 MHz.**

- `62 500 000` → QPLL0 in bolla ✅
- `63 476 562` → FBDIV a 65 invece di 66 (da correggere via DRP)
- `0` → clock morto

⚠️ Niente `devmem2` qui: su aarch64 fa accessi a 64 bit e va in SIGBUS sugli offset
`addr%8==4` (ci abbiamo perso due giorni). Questa cella usa mmap a **32 bit**.

In [ ]:
hz, cnt, gate_s = freq_meas()
print(f"gate            = {gate_s:.3f} s")
print(f"FREQ_CNT        = {cnt:,}".replace(",", " "))
print(f"→ TXUSRCLK2     = {hz/1e6:.4f} MHz")

err_ppm = (hz - 62.5e6) / 62.5e6 * 1e6
if abs(err_ppm) < 500:
    print(f"\n✅ QPLL0 SANA — scarto {err_ppm:+.1f} ppm dai 62.5 MHz nominali")
elif cnt == 0:
    print("\n❌ conteggio 0 → clock morto (TX in reset? QPLL0 non locka?)")
else:
    print(f"\n⚠️ fuori tolleranza: {err_ppm:+.0f} ppm — controllare FBDIV/SDM")

## 7 — Monitor del lock: si arriva a `TRACK_PHASE`? e ci si resta?

Legge in loop `stat` + `pll stat` e disegna **`cko`** (clock offset dal master, in ps) e
**`md`** (uscita del DAC che sterza l'SDM).

**Obiettivo:** `MFL1` **`MPL1`**, `ss = TRACK_PHASE`, **`cko` entro pochi ps**.
Sulla scheda: **LED1 fisso** (cage) e **UF1 che lampeggia 1/s** (PPS).

### Tenuta del lock — `DelCnt` e `m_phase_lock_ms`

- **`DelCnt`** = contatore dei **delock** (`softpll.delock_count`, `softpll_ng.c:203`).
  È **cumulativo da `ptp start`** e non si azzera da solo → quello che conta non è il
  valore assoluto ma il suo **incremento**: il monitor salva la baseline alla prima
  lettura e segna ogni crescita come **riga rossa tratteggiata** sui grafici.
- Quando scatta, la FSM torna a **`SEQ_CLEAR_DACS`**: azzera i DAC e rifà tutto il
  bring-up (helper → main → fase), ≈ **7 s di sincronizzazione persa**. Non è un
  ri-aggancio morbido.
- **`m_phase_lock_ms`** = **quanto ci ha messo** il main ad agganciare la fase, contato
  da `mpll_start()` (`spll_main.c:236` → `:428`). ⚠️ **Non** è da quanto tempo il lock
  regge: è la **latenza di aggancio**. Viene riscritto a ogni nuovo aggancio, quindi se
  cambia fra due letture il nodo ha ri-agganciato.

⚠️ A far scattare il delock è la **fase**, non la frequenza: in `spll_main.c:153`
`s->locked` segue solo `phase_ld`. Nota che **`MFL` non torna mai a 0** una volta
salito (`freq_ld.delock_samples = 20000 > lock_samples = 50` → il rilevatore non può
sganciare), quindi `MFL1` dice solo che *a un certo punto* la frequenza era agganciata.

Riferimento: nel test del 13/7 con v15 → `DelCnt:0` su tutte le letture, 8 minuti.

In [ ]:
# NB: la scheda ha matplotlib 3.5.1 (sistema) ma matplotlib_inline 0.2.2 (~/.local),
# che pretende RcParams._get → il backend "inline" di Jupyter esplode.
# Aggiriamo: rendering con Agg + figure mostrate come PNG. Nessuna dipendenza nuova.
import sys, io
sys.modules.pop("matplotlib.pyplot", None)
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from IPython.display import clear_output, display, Image

def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
    plt.close(fig)
    display(Image(data=buf.getvalue()))

def parse_pll(txt):
    """Campi numerici k:v di 'pll stat' (DelCnt, m_phase_lock_ms, setp, ...)."""
    return dict(re.findall(r"([A-Za-z_]\w*):(-?\d+)", txt))

N = 25                       # letture (~10 s l'una)
t, cko_h, md_h = [], [], []
del0 = del_prev = None       # DelCnt e' cumulativo da 'ptp start' → serve una baseline
delock_at = []               # letture in cui DelCnt e' cresciuto

for i in range(N):
    s  = parse_stat(stat())
    ps = wrc("pll stat", echo=False)
    p  = parse_pll(ps)
    mfl = "MFL1" if "MFL1" in ps else "MFL0"
    mpl = "MPL1" if "MPL1" in ps else "MPL0"

    dc  = int(p["DelCnt"]) if "DelCnt" in p else None
    plm = int(p["m_phase_lock_ms"]) if "m_phase_lock_ms" in p else None

    # Ogni incremento di DelCnt = la FSM del softpll e' ripartita da SEQ_CLEAR_DACS:
    # azzera i DAC e ri-aggancia tutto da capo (~7 s di sincronizzazione persa).
    nuovo_delock = False
    if dc is not None:
        if del0 is None:
            del0 = dc
        nuovo_delock = del_prev is not None and dc > del_prev
        if nuovo_delock:
            delock_at.append(i)
        del_prev = dc

    if s:
        t.append(i)
        cko_h.append(int(s.get("cko", 0)))
        md_h.append(int(s.get("md", 0)))

    clear_output(wait=True)
    ok = (mpl == "MPL1" and s.get("ss") == "TRACK_PHASE")
    print(f"[{i+1:2d}/{N}] {datetime.now():%H:%M:%S}   {'🏆 PHASE LOCK' if ok else '⏳ in aggancio…'}")
    print(f"   lnk:{s.get('lnk','?')}  lock:{s.get('lock','?')}  ptp:{s.get('ptp','?')}  "
          f"ss:{s.get('ss','?')}")
    print(f"   {mfl} {mpl}   cko:{s.get('cko','?')} ps   md:{s.get('md','?')}   "
          f"setp:{s.get('setp','?')}")
    # m_phase_lock_ms = LATENZA di aggancio (da mpll_start alla presa di fase,
    # spll_main.c:428), non da quanto regge il lock. Cambia a ogni nuovo aggancio.
    d_str = "?" if dc is None else f"{dc}  (+{dc - del0} da inizio monitor)"
    print(f"   DelCnt:{d_str}   m_phase_lock_ms:{plm if plm is not None else '?'}")
    if nuovo_delock:
        print("   ⚠️  DELOCK: fase persa, il softpll sta ri-agganciando da capo")

    if len(t) > 1:
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
        ax[0].plot(t, cko_h, marker="o", ms=3)
        ax[0].axhline(0, lw=.8, color="#888")
        ax[0].set_title("cko — clock offset dal master [ps]"); ax[0].set_xlabel("lettura")
        ax[0].grid(alpha=.3)
        ax[1].plot(t, md_h, marker="o", ms=3, color="#D55E00")
        ax[1].set_title("md — uscita DAC (steering SDM)"); ax[1].set_xlabel("lettura")
        ax[1].grid(alpha=.3)
        for k, x in enumerate(delock_at):        # ogni delock = riga rossa su entrambi
            for a in ax:
                a.axvline(x, color="#C00", ls="--", lw=1.2,
                          label="delock" if k == 0 and a is ax[0] else None)
        if delock_at:
            ax[0].legend(loc="best", fontsize=8)
        fig.tight_layout()
        show(fig)

    time.sleep(8)

print("\nfine monitor.")
if cko_h:
    tail = cko_h[-10:]
    print(f"cko sulle ultime {len(tail)} letture: min {min(tail)} / max {max(tail)} ps")
if del0 is None:
    print("DelCnt non letto: 'pll stat' non ha risposto (vUART occupata da un altro consumatore?)")
elif del_prev == del0:
    print(f"DelCnt fermo a {del0} → nessun delock durante il monitor ✅")
else:
    print(f"⚠️  DelCnt {del0} → {del_prev}: {del_prev - del0} delock, alle letture {delock_at}")

## 8 — La GUI del WRPC dentro il notebook

Il firmware ha il comando **`gui`**: la classica schermata di monitor del WR PTP Core
(tempo TAI, stato PPSI, servo, parametri di timing). Qui la catturiamo dalla vUART per
**30 s**, la renderizziamo live e la **fermiamo da soli** inviando `q`.

Non è testo lineare: il `gui` pilota un terminale con **posizionamento assoluto del cursore**,
quindi la cella contiene un mini-emulatore VT100 (griglia di caratteri + colori ANSI → HTML).

**Tre trappole, tutte già gestite qui dentro:**
1. **`stat` nudo è un TOGGLE** dell'output continuo: se è acceso, il suo flusso si sovrappone
   al `gui` e lo cancella. La cella verifica che la linea sia silenziosa e, se non lo è, lo spegne.
2. **`ESC[J` ≠ `ESC[2J`**: il `gui` disegna le etichette **una sola volta** e poi ridipinge solo i
   valori usando `ESC[J` (= cancella *dal cursore in giù*). Trattarlo come clear-screen totale
   fa sparire tutte le etichette.
3. Il `gui` va avanti finché non premi un tasto → se non mandi `q` continua a sputare byte e
   **inquina ogni lettura successiva** della vUART.

In [ ]:
# import completi: la cella dev'essere autosufficiente anche se salti la 7
from IPython.display import HTML, display, clear_output

ROWS, COLS = 26, 90
# SGR ANSI → colori CSS (il gui usa 01;3x = bold + colore)
_FG = {"30":"#555","31":"#e06c75","32":"#98c379","33":"#e5c07b",
       "34":"#61afef","35":"#c678dd","36":"#56b6c2","37":"#e6e6e6","39":"#e6e6e6"}

class Screen:
    """Mini-VT100: interpreta il sottoinsieme ANSI usato dal gui del WRPC."""
    def __init__(self):
        self._wipe(); self.last = None; self.pend = b""
    def _wipe(self):
        self.buf = [[(" ", "")] * COLS for _ in range(ROWS)]
        self.buf = [[(" ", "") for _ in range(COLS)] for _ in range(ROWS)]
        self.r = self.c = 0; self.sgr = ""
    def feed(self, data):
        data = self.pend + data; self.pend = b""
        i, n = 0, len(data)
        while i < n:
            b = data[i]
            if b == 0x1b:
                if i + 1 >= n:
                    self.pend = data[i:]; return
                if data[i+1] == ord("["):
                    j = i + 2
                    while j < n and data[j] not in b"JHfKmABCD":
                        j += 1
                    if j >= n:
                        self.pend = data[i:]; return      # CSI spezzata fra due letture
                    self._csi(data[i+2:j].decode("ascii","replace"), chr(data[j]))
                    i = j + 1; continue
            ch = chr(b)
            if ch == "\n":   self.r += 1; self.c = 0
            elif ch == "\r": self.c = 0
            elif b >= 32:
                if self.r < ROWS and self.c < COLS:
                    self.buf[self.r][self.c] = (ch, self.sgr)
                self.c += 1
                if self.c >= COLS: self.c = 0; self.r += 1
            self.r = min(self.r, ROWS - 1)
            i += 1
    def _csi(self, params, final):
        if final == "J":
            # ESC[2J = schermo intero; ESC[J / ESC[0J = solo dal cursore in giù.
            # Il gui usa il secondo a ogni refresh: confonderli cancella le etichette.
            if params == "2":
                if any(ch != " " for row in self.buf for ch, _ in row):
                    self.last = self.buf
                self._wipe()
            elif params in ("", "0"):
                for c in range(self.c, COLS): self.buf[self.r][c] = (" ", "")
                for r in range(self.r + 1, ROWS):
                    self.buf[r] = [(" ", "") for _ in range(COLS)]
        elif final in "Hf":
            p = [int(x) if x.isdigit() else 1 for x in (params.split(";") + ["1","1"])[:2]]
            self.r, self.c = max(0, p[0]-1), max(0, p[1]-1)
        elif final == "K":
            for c in range(self.c, COLS): self.buf[self.r][c] = (" ", "")
        elif final == "m":
            self.sgr = params
    def html(self):
        buf = self.buf
        if not any(ch != " " for row in buf for ch, _ in row):
            buf = self.last or buf                     # frame appena azzerato
        out = []
        for row in buf:
            line, cur, run = [], None, []
            for ch, sgr in row:
                col = _FG.get(sgr.split(";")[-1], "#e6e6e6") if sgr else "#e6e6e6"
                bold = "font-weight:600;" if sgr.startswith("01") else ""
                key = (col, bold)
                if key != cur and run:
                    line.append(f'<span style="color:{cur[0]};{cur[1]}">{"".join(run)}</span>'); run = []
                cur = key; run.append(ch.replace("&","&amp;").replace("<","&lt;"))
            if run:
                line.append(f'<span style="color:{cur[0]};{cur[1]}">{"".join(run)}</span>')
            out.append("".join(line).rstrip())
        body = "\n".join(out).rstrip()
        return HTML('<pre style="background:#1e1e1e;color:#e6e6e6;padding:12px;'
                    'border-radius:6px;line-height:1.25;font-size:12.5px;'
                    f'overflow-x:auto">{body}</pre>')

def _raw(sec):
    """Lettura GREZZA della vUART: il gui manda byte di escape, non testo →
    _drain() (che decodifica in ascii) li rovinerebbe."""
    out, t0 = bytearray(), time.time()
    while time.time() - t0 < sec:
        v = rd(_wb, RDR)
        if v & 0x100: out.append(v & 0xff)
        else: time.sleep(0.002)
    return bytes(out)

def _send_raw(s):
    for ch in s.encode():
        while not (rd(_wb, TDR) & 0x100): pass
        wr(_wb, TDR, ch)

def _quiet():
    """'stat' nudo è un TOGGLE: se il flusso continuo è acceso, si mangia il gui → spegnilo."""
    if len(_raw(1.2)) > 50:
        print("… flusso 'stat' continuo acceso: lo spengo (è un toggle)")
        _send_raw("stat\r"); time.sleep(0.5); _raw(2.0)

def wrc_gui(seconds=30, fps=1.0):
    """Mostra la GUI del WRPC per N secondi, poi la chiude con 'q'."""
    ensure_attached()      # si aggancia da sola: la cella 2 (che ricarica la PL) non serve
    _quiet()
    _send_raw("gui\r")
    scr, t0 = Screen(), time.time()
    try:
        while time.time() - t0 < seconds:
            scr.feed(_raw(1.0 / fps))
            clear_output(wait=True)
            display(scr.html())
            print(f"⏱️  {int(time.time()-t0):2d}/{seconds}s — chiudo da solo con 'q'")
    finally:
        _send_raw("q")                        # SEMPRE, anche se interrompi il kernel
        time.sleep(0.4); _raw(1.5)            # svuota la coda del gui
    print("✅ gui chiusa, vUART di nuovo libera per gli altri comandi")

wrc_gui(30)

## 9 — Console libera

Cella riusabile per interrogare il nodo: `pll stat`, `stat`, `mdio dump`, `ptp`, `time`…

In [ ]:
wrc("pll stat", hard=4.0);

## 10 — build_id: cosa gira davvero sulla scheda (richiede v16+)

**⚠️ Richiede il firmware v16 o successivo caricato.** Su v15 questi indirizzi non hanno
una periferica dietro nel crossbar: leggerli genera un bus error che **pianta il kernel
Python** (va riavviato da Jupyter). La cella sotto controlla da sola quale app è attiva
e si rifiuta di proseguire se non è una v16+.

Ogni build imbusta nel bitstream l'hash git del gateware **e** di `wrpc-sw` (il softcore
LM32) al momento della synth — così si può rispondere a "questa scheda sta girando
quello che penso io?" leggendo un registro, senza fidarsi della memoria.

In [ ]:
# si rifiuta di leggere 0xA0050000 se non c'e' un'app v16+ attiva (vedi markdown sopra)
_listapps = sh("sudo xmutil listapps 2>&1")
active_app = None
for line in _listapps.splitlines():
    parts = line.split()
    if len(parts) >= 2 and parts[-1].rstrip(",") not in ("-1", "Active_slot"):
        active_app = parts[0]

print(f"app attiva: {active_app!r}")
assert active_app and re.search(r"_v(1[6-9]|[2-9]\d)\b", active_app), (
    f"Serve un'app v16+ caricata (trovata: {active_app!r}). "
    f"Leggere 0xA0050000/0xA0060000 senza la periferica dietro pianta il kernel "
    f"— fermo qui apposta, non e' un bug.")

BID_BASE, BID_SPAN = 0xA0050000, 0x1000
_bid = _map(BID_BASE, BID_SPAN)

fw, sw, ts, flags = rd(_bid, 0x00), rd(_bid, 0x04), rd(_bid, 0x08), rd(_bid, 0x0C)
ver      = (flags >> 24) & 0xFF
sw_dirty = (flags >> 1) & 1
fw_dirty = flags & 1

print(f"\nFW_GITHASH  0x{fw:08x}   (repo sfp_drp_kr260_sdm)")
print(f"SW_GITHASH  0x{sw:08x}   (wrpc-sw)")
print(f"BUILD_TS    {datetime.fromtimestamp(ts):%Y-%m-%d %H:%M:%S}   (0x{ts:08x})")
print(f"FLAGS       0x{flags:08x}   → versione {ver}"
      f"{'  ⚠️ FW DIRTY' if fw_dirty else ''}{'  ⚠️ SW DIRTY' if sw_dirty else ''}")

if (fw >> 16) == 0xDEAD or (sw >> 16) == 0xDEAD:
    print("\n⚠️  Hash di default (0xDEADxxxx): build fatta SALTANDO l'iniezione git "
          "— rebuild_v16.tcl non e' girato correttamente su questa immagine.")

print("\nConfronta con (su pcbetif02):")
print("  git -C ~/sfp_drp_kr260_sdm log --oneline -1")
print("  git -C ~/wrpc-sw log --oneline -1")

## 11 — TDC a catena di carry: monitor e calibrazione (richiede v16+)

TDC costruito con una linea di ritardo su catena di **CARRY8** (UltraScale+), campionata
a **375 MHz** (bin coarse 2,667 ns) da un MMCM riferito a **TXUSRCLK2**, il clock
WR-disciplinato — la stessa base tempi del frequenzimetro. Il numero di tap si legge dal
registro `TAPS` (dinamico: 768 fino al 21/7/2026, **1536** da quando la catena è stata
raddoppiata — vedi nota 11b).

| Ingresso (`input_sel`) | Cosa fa |
|---|---|
| `0` | PMOD1 pin fisico 3 (E10): il segnale esterno da timestampare |
| `1` | **PPS del WRPC** — ancora coarse/fine al confine di secondo TAI (richiede `TRACK_PHASE`) |
| `2` | **ring oscillator libero**: nessun PLL, per il code-density test di calibrazione |
| `3` | strobe software (un fronte per ogni scrittura di CSR con bit4=1) |

⚠️ Come per la sezione 10, richiede v16+ caricato. La cella seguente riusa la stessa
verifica `active_app` di prima (rieseguila se hai riavviato il kernel).

In [ ]:
# cella autosufficiente: riverifica active_app anche se hai saltato la 10
if "active_app" not in dir() or not active_app:
    _listapps = sh("sudo xmutil listapps 2>&1")
    active_app = None
    for line in _listapps.splitlines():
        parts = line.split()
        if len(parts) >= 2 and parts[-1].rstrip(",") not in ("-1", "Active_slot"):
            active_app = parts[0]
assert active_app and re.search(r"_v(1[6-9]|[2-9]\d)\b", active_app), (
    f"Serve un'app v16+ caricata (trovata: {active_app!r}) — vedi sezione 10.")

TDC_BASE, TDC_SPAN = 0xA0060000, 0x1000
_tdc = _map(TDC_BASE, TDC_SPAN)
CSR, STATUS, TSLO, TSHI, TAPS, CLKHZ, CNTHITS, RINGCNT = \
    0x00, 0x04, 0x08, 0x0C, 0x10, 0x14, 0x18, 0x1C

def tdc_reset():
    """Azzera FIFO + sticky overflow. NON tocca il coarse counter (free-running)."""
    wr(_tdc, CSR, 0x08); time.sleep(0.01); wr(_tdc, CSR, 0x00)

def tdc_set(enable, sel):
    """sel: 0=PMOD 1=PPS 2=ring 3=sw. Cambiare sel con enable=0 evita fronti spuri."""
    wr(_tdc, CSR, ((sel & 3) << 1) | (1 if enable else 0))

def tdc_status():
    st = rd(_tdc, STATUS)
    return dict(locked=st & 1, empty=(st >> 1) & 1, ovfl=(st >> 2) & 1,
                fifo_count=(st >> 16) & 0x3FF)

# fine e' passato da 10 a 11 bit il 21/7/2026 (catena raddoppiata a 1536 tap).
# Layout nel registro a 64 bit: {seq(8), pad(12-fbits), coarse(44), fine(fbits)}
# -> TS_LO = bit[31:0] = {coarse_basso(32-fbits), fine(fbits)}
#    TS_HI = bit[63:32] = {seq(8), pad(12-fbits), coarse_alto(12+fbits)}
# La maschera si adatta da sola leggendo TAPS, cosi' la cella funziona anche
# su immagini piu' vecchie con la catena a 768 (fbits=10).
def tdc_pop():
    """Un timestamp dalla FIFO: (seq, coarse, fine), o None se vuota."""
    if tdc_status()["empty"]:
        return None
    lo, hi = rd(_tdc, TSLO), rd(_tdc, TSHI)          # leggere LO poi HI: LO fa il pop
    fbits  = 11 if taps > 1023 else 10
    fine   = lo & ((1 << fbits) - 1)
    coarse = ((hi & ((1 << (12 + fbits)) - 1)) << (32 - fbits)) | (lo >> fbits)
    seq    = (hi >> 24) & 0xFF
    return seq, coarse, fine

def tdc_ring_hz(gate_s=1.0):
    """Frequenza indicativa del ring oscillator (deve girare: tdc_set(1, 2) prima)."""
    r0 = rd(_tdc, RINGCNT); t0 = time.time()
    time.sleep(gate_s)
    r1 = rd(_tdc, RINGCNT)
    return ((r1 - r0) & 0xFFFFFFFF) / (time.time() - t0)

taps, clk_hz = rd(_tdc, TAPS), rd(_tdc, CLKHZ)
print(f"TDC: {taps} tap, clock {clk_hz/1e6:.1f} MHz (bin coarse = {1e9/clk_hz:.3f} ns)")
print(f"STATUS: {tdc_status()}")

### 11a — Autotest sul PPS

Richiede `TRACK_PHASE` (sezione 7). Raccoglie qualche impulso PPS e verifica che gli
intervalli tra un fronte e il successivo siano **375 000 000 conteggi coarse** = 1 s
esatto, alla risoluzione del bin (2,667 ns). Verificato a mano il 21/7/2026: 0 ppm di
errore su 5 intervalli consecutivi.

In [ ]:
print("=== Autotest PPS: raccolgo 6 impulsi, atteso 1s esatto tra due successivi ===")
tdc_reset()
tdc_set(1, 1)   # enable, sel=PPS
t0 = time.time()
while tdc_status()["fifo_count"] < 6 and time.time() - t0 < 12:
    time.sleep(0.2)

stamps = []
while True:
    s = tdc_pop()
    if s is None:
        break
    stamps.append(s)
tdc_set(0, 0)

st = tdc_status()
print(f"raccolti {len(stamps)} stamp — mmcm_locked={st['locked']}  overflow={st['ovfl']}")

if len(stamps) < 2:
    print("⚠️  Troppo pochi stamp: il WRPC è davvero in TRACK_PHASE? "
          "(sezione 7, o `wrc('pll stat')` in sezione 9)")
else:
    deltas = [stamps[i+1][1] - stamps[i][1] for i in range(len(stamps)-1)]
    err_ppm = [(d - clk_hz) / clk_hz * 1e6 for d in deltas]
    for i, (d, e) in enumerate(zip(deltas, err_ppm)):
        print(f"  intervallo {i+1}: {d:,} conteggi coarse   ({e:+.3f} ppm rispetto a 1s)"
              .replace(",", " "))
    peak = max(abs(e) for e in err_ppm)
    print("✅ PPS agganciato al TDC" if peak < 1 else f"⚠️ scarto anomalo: {peak:.3f} ppm")

### 11b — Calibrazione code-density (ring oscillator)

Il ring oscillator (nessun PLL, solo ritardo di gate — dipende da temperatura/tensione)
alimenta `input_sel=2`: genuinamente incommensurabile col clock del TDC. I fronti
dovrebbero coprire in modo uniforme tutti i tap.

**Storia**: il 20-21/7/2026 tre ipotesi diverse sono state verificate e **smentite** sul
ferro senza cambiare l'istogramma (sempre incollato al soffitto della catena a 768 tap):
sorgente non incommensurabile, doppio stadio di registrazione, polarità invertita di
`CARRY8` (quest'ultima un bug reale, confermato via simulazione, ma non la causa dello
spread). **Risolto il 21/7 pomeriggio raddoppiando la catena a 1536 tap** (`N_C8=192`):
range passato da 549-768 (incollato al massimo) a **605-1203** (centrato, con margine su
entrambi i lati) — la firma di una catena che finalmente copre più tempo del periodo di
campionamento (2,667 ns). Dettagli in `V16_BUILD_ID_TDC.md`.

In [ ]:
# stesso blocco Agg/show della sezione 7, ripetuto qui per rendere la cella
# autosufficiente anche se salti quella sezione
if "show" not in dir():
    import sys, io
    sys.modules.pop("matplotlib.pyplot", None)
    import matplotlib
    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt
    from IPython.display import Image, display

    def show(fig):
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
        plt.close(fig)
        display(Image(data=buf.getvalue()))
else:
    import matplotlib.pyplot as plt

print("=== Calibrazione code-density: ring oscillator libero (input_sel=2) ===")
tdc_reset()
tdc_set(1, 2)   # enable, sel=ring — SENZA questo il ring resta spento (visto il 21/7:
                # tdc_reset() disabilita, tdc_ring_hz()/tdc_pop() letti a vuoto -> 0 Hz, 0 stamp)
hz = tdc_ring_hz(0.3)
print(f"ring oscillator: ~{hz/1e6:.1f} MHz")

DURATION_S = 10
hist = [0] * (taps + 1)
n = 0
t_end = time.time() + DURATION_S
while time.time() < t_end:
    s = tdc_pop()
    if s is None:
        continue
    _, _, fine = s
    hist[fine] += 1
    n += 1
tdc_set(0, 0)

print(f"stamp raccolti: {n}  (in {DURATION_S}s)")

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(range(len(hist)), hist, width=1.0, color="#61afef")
ax.set_xlabel(f"tap (0-{taps})"); ax.set_ylabel("conteggi")
ax.set_title("Code-density del TDC — deve coprire l'intero range se la cattura è corretta")
ax.grid(alpha=.3)
show(fig)

# Il criterio giusto NON è "copre quasi tutto il range" (0-taps): la finestra attiva
# è fisiologicamente solo una FRAZIONE della catena (dipende dal rapporto fra tempo di
# transito totale e periodo di campionamento). Il criterio giusto è che NON tocchi gli
# estremi (0 o taps): se ci arriva incollato, la catena è di nuovo troppo corta (o un
# bug l'ha rotta di nuovo) — vedi V16_BUILD_ID_TDC.md per la storia completa del 20-21/7.
nz = [i for i, c in enumerate(hist) if c > 0]
if nz:
    lo_r, hi_r = min(nz), max(nz)
    margin_lo, margin_hi = lo_r, taps - hi_r
    print(f"tap distinti colpiti: {len(nz)}   range: [{lo_r}, {hi_r}]   "
          f"margine: {margin_lo} sotto, {margin_hi} sopra")
    if margin_lo < taps * 0.05 or margin_hi < taps * 0.05:
        print("⚠️  Range incollato a un estremo (margine <5%) — catena di nuovo troppo")
        print("   corta rispetto al periodo di campionamento, o uno dei bug già visti")
        print("   (sorgente, doppio registro, polarità CARRY8) è tornato. Vedi")
        print("   V16_BUILD_ID_TDC.md prima di riprovare le stesse ipotesi.")
    else:
        print("✅ range centrato, margine su entrambi i lati — cattura sana.")
        print("   Prossimo passo: sezione 12 per la calibrazione vera (ps/tap).")
else:
    print("⚠️  Nessuno stamp raccolto — enable/sel sono giusti? mmcm_locked?")

## 12 — Calibrazione vera: da tap a picosecondi (richiede v16+)

La sezione 11b mostra *che* la catena copre un range sano; questa sezione lo trasforma
in una **tabella di calibrazione** usabile per correggere i timestamp reali.

**Teoria (code-density, in breve)**: il ring oscillator è asincrono rispetto al clock
del TDC, quindi la fase di arrivo vera `τ` (tempo fra l'inizio del periodo di
campionamento e il fronte vero) è **uniforme** su `[0, T_periodo)`. Poiché `fine` è una
funzione monotona di `τ` (più tap raggiunti = fronte arrivato prima nel periodo = più
tempo per propagarsi), la densità di conteggi in ciascun bin è proporzionale alla
**larghezza reale in tempo** di quel bin:

```
width_ps[f]  = T_periodo_ps × hist[f] / totale_conteggi
delta_ps[f]  = T_periodo_ps × (conteggi con fine < f) / totale_conteggi   (funzione di correzione, monotona crescente)
```

`delta_ps[f]` è quanto sottrarre a `coarse × T_periodo_ps` per ottenere il tempo vero
del fronte: `t_ps = coarse × T_periodo_ps − delta_ps[fine]`.

**Verifica di autoconsistenza integrata nella cella**: la somma delle `width_ps` sui bin
attivi deve tornare vicina a `T_periodo_ps` (2666,667 ps a 375 MHz) — se non torna,
qualcosa nella raccolta è andato storto (troppo pochi campioni, overflow FIFO non
gestito, ecc.), non fidarsi della calibrazione prodotta.

Verificato il 21/7/2026: somma = 2666.667 ps esatta, funzione monotona, correzioni reali
nell'ordine di centinaia-migliaia di ps (piena scala del periodo).

**Deriva termica**: i ritardi di propagazione in fabric dipendono dalla temperatura —
questa curva è valida finché la temperatura non cambia troppo. Ogni esecuzione di
questa cella **salva automaticamente** timestamp + temperatura PS/PL (dal SYSMON
interno, via IIO — nessuna periferica nuova, già esposto da Linux) + tabella di
calibrazione in `/home/ubuntu/tdc_calibration_log.json`. Rilanciarla più volte nel
tempo costruisce lo storico che la **sezione 13** usa per stimare la deriva reale
ps/°C di questa scheda.

In [ ]:
# cella autosufficiente: se hai saltato la 11, riaggancia TDC_BASE/tdc_* qui
if "tdc_pop" not in dir():
    if "active_app" not in dir() or not active_app:
        _listapps = sh("sudo xmutil listapps 2>&1")
        active_app = None
        for line in _listapps.splitlines():
            parts = line.split()
            if len(parts) >= 2 and parts[-1].rstrip(",") not in ("-1", "Active_slot"):
                active_app = parts[0]
    assert active_app and re.search(r"_v(1[6-9]|[2-9]\d)\b", active_app), (
        f"Serve un'app v16+ caricata (trovata: {active_app!r}) — vedi sezione 10.")
    print("⚠️  Sezione 11 non eseguita in questa sessione: rieseguila prima "
          "(qui servono solo taps/clk_hz/tdc_*, ma è più pulito passare da lì).")
    raise RuntimeError("esegui prima la sezione 11")

import json, os, glob

TDC_CAL_LOG = "/home/ubuntu/tdc_calibration_log.json"

def tdc_read_temps():
    """PS/PL/remote dal SYSMON (AMS), via IIO — nessuna periferica PL aggiuntiva,
    è già esposto da Linux su tutte le immagini (iio:device0, canale 'ams')."""
    base = "/sys/bus/iio/devices/iio:device0"
    out = {}
    for name in ("ps_temp", "pl_temp", "remote_temp"):
        raw_f = glob.glob(f"{base}/in_temp*_{name}_raw")[0]
        prefix = raw_f[:-len("_raw")]
        raw    = int(open(raw_f).read())
        scale  = float(open(prefix + "_scale").read())
        offset = int(open(prefix + "_offset").read())
        out[name] = (raw + offset) * scale / 1000.0
    return out

def tdc_log_calibration(lo_r, hi_r, delta_ps, n):
    """Appende la calibrazione corrente a TDC_CAL_LOG (temperatura + tabella
    ristretta al range attivo, per confrontare più calibrazioni nel tempo)."""
    temps = tdc_read_temps()
    entry = {
        "ts": datetime.now().isoformat(timespec="seconds"),
        "pl_temp_c": round(temps["pl_temp"], 2),
        "ps_temp_c": round(temps["ps_temp"], 2),
        "n_samples": n, "lo_r": lo_r, "hi_r": hi_r,
        "delta_ps_active": [round(v, 3) for v in delta_ps[lo_r:hi_r + 1]],
    }
    log = []
    if os.path.exists(TDC_CAL_LOG):
        try:
            log = json.load(open(TDC_CAL_LOG))
        except Exception:
            log = []
    log.append(entry)
    json.dump(log, open(TDC_CAL_LOG, "w"), indent=1)
    print(f"Temperatura: PL={temps['pl_temp']:.2f}°C  PS={temps['ps_temp']:.2f}°C")
    print(f"Calibrazione salvata in {TDC_CAL_LOG} ({len(log)} voci nello storico)")
    return entry

TDC_CAL_DURATION_S = 30      # più lunga della 11b: serve buona statistica per bin

print(f"=== Calibrazione: raccolta {TDC_CAL_DURATION_S}s col ring oscillator ===")
tdc_reset()
hist = [0] * (taps + 1)
n = 0
t_end = time.time() + TDC_CAL_DURATION_S
tdc_set(1, 2)   # enable, sel=ring
while time.time() < t_end:
    s = tdc_pop()
    if s is None:
        continue
    _, _, fine = s
    hist[fine] += 1
    n += 1
tdc_set(0, 0)

nz = [i for i, c in enumerate(hist) if c > 0]
if not nz or n < 10_000:
    print(f"⚠️  Raccolti solo {n} stamp ({len(nz)} bin) — troppo pochi per una "
          f"calibrazione affidabile. Il ring gira? (sezione 11: tdc_ring_hz())")
else:
    lo_r, hi_r = min(nz), max(nz)
    T_ps = 1e12 / clk_hz

    # --- tabella di calibrazione: width_ps e delta_ps (cumulativa, monotona) ---
    delta_ps = [0.0] * (taps + 1)
    cum = 0
    for f in range(taps + 1):
        delta_ps[f] = T_ps * cum / n
        cum += hist[f]
    width_ps = [T_ps * hist[f] / n for f in nz]

    print(f"stamp raccolti: {n}   range attivo: [{lo_r}, {hi_r}]   bin attivi: {len(nz)}")
    print(f"width_ps: min={min(width_ps):.3f}  media={sum(width_ps)/len(width_ps):.3f}  "
          f"max={max(width_ps):.3f}  (media attesa se uniforme: {T_ps/len(nz):.3f})")
    somma = sum(width_ps)
    print(f"autoconsistenza: Σwidth_ps = {somma:.3f} ps  (atteso ≈ T_periodo = {T_ps:.3f} ps, "
          f"scarto {abs(somma-T_ps)/T_ps*100:.2f}%)")

    def tdc_calibrated_ps(coarse, fine):
        """Tempo assoluto in ps (origine arbitraria, legata al coarse counter),
        con la correzione fine applicata. fine fuori dal range calibrato -> None."""
        if fine < lo_r or fine > hi_r:
            return None
        return coarse * T_ps - delta_ps[fine]

    tdc_log_calibration(lo_r, hi_r, delta_ps, n)

    # grafico della curva di calibrazione (sanity check visivo: deve essere liscia
    # e monotona crescente all'interno del range attivo)
    if "show" not in dir():
        import sys, io
        sys.modules.pop("matplotlib.pyplot", None)
        import matplotlib
        matplotlib.use("Agg", force=True)
        import matplotlib.pyplot as plt
        from IPython.display import Image, display
        def show(fig):
            buf = io.BytesIO()
            fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
            plt.close(fig)
            display(Image(data=buf.getvalue()))
    else:
        import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.plot(range(lo_r, hi_r + 1), delta_ps[lo_r:hi_r + 1], lw=1.2, color="#98c379")
    ax.set_xlabel("tap (fine)"); ax.set_ylabel("delta_ps  (correzione)")
    ax.set_title("Curva di calibrazione — deve essere liscia e monotona crescente")
    ax.grid(alpha=.3)
    show(fig)

    # demo pratica: qualche stamp fresco, grezzo vs calibrato
    tdc_reset(); tdc_set(1, 2); time.sleep(0.05)
    demo, t_end = [], time.time() + 0.5
    while time.time() < t_end and len(demo) < 5:
        s = tdc_pop()
        if s:
            demo.append(s)
    tdc_set(0, 0)
    print("\ndemo — (seq, coarse, fine) -> grezzo vs calibrato:")
    for seq, coarse, fine in demo:
        raw_ps = coarse * T_ps
        cal_ps = tdc_calibrated_ps(coarse, fine)
        corr   = raw_ps - cal_ps if cal_ps is not None else float("nan")
        print(f"  seq={seq:3d} coarse={coarse} fine={fine:4d}   "
              f"correzione={corr:+8.1f} ps")

    print("\n✅ tdc_calibrated_ps(coarse, fine) pronta per l'uso su qualunque "
          "sorgente (PMOD, PPS, sw) finché non si ricarica la PL.")
    print("   Rilancia questa cella più tardi (temperatura diversa) e poi la "
          "sezione 13 per vedere la deriva reale ps/°C.")

## 13 — Deriva della calibrazione: quanto conta la temperatura?

Legge `/home/ubuntu/tdc_calibration_log.json` (costruito dalla sezione 12, una voce per
ogni esecuzione) e confronta la calibrazione più fredda con la più calda salvate finora,
sull'intersezione dei loro range attivi. Serve **almeno una seconda calibrazione a
temperatura diversa**: la prima volta questa cella si limita a dirlo.

**Uso pratico**: rilancia la sezione 12 periodicamente durante una sessione lunga (appena
dopo il load, e poi ogni 15-30 minuti, o quando sai che la temperatura ambiente/del
carico è cambiata) — ogni run si accumula nello storico senza bisogno di altro.

In [ ]:
# cella autosufficiente: ridefinisce TDC_CAL_LOG anche se hai saltato la 12
import json, os
if "TDC_CAL_LOG" not in dir():
    TDC_CAL_LOG = "/home/ubuntu/tdc_calibration_log.json"

if not os.path.exists(TDC_CAL_LOG):
    log = []
else:
    log = json.load(open(TDC_CAL_LOG))

print(f"{len(log)} calibrazioni salvate in {TDC_CAL_LOG}")
for i, e in enumerate(log):
    print(f"  [{i}] {e['ts']}  PL={e['pl_temp_c']:.2f}°C  "
          f"range=[{e['lo_r']},{e['hi_r']}]  n={e['n_samples']}")

if len(log) < 2:
    print("\n⚠️  Serve almeno una seconda calibrazione (sezione 12) a temperatura "
          "diversa per vedere una deriva. Rilancia più tardi e riesegui questa cella.")
else:
    # intersezione dei range attivi fra tutte le voci salvate
    lo = max(e["lo_r"] for e in log)
    hi = min(e["hi_r"] for e in log)
    if hi <= lo:
        print("\n⚠️  I range attivi delle calibrazioni salvate non si sovrappongono "
              "abbastanza da poter confrontare — probabile che una sia anomala "
              "(vedi sezione 11b: non deve toccare gli estremi).")
    else:
        temps_c = [e["pl_temp_c"] for e in log]
        i_cold = min(range(len(log)), key=lambda i: temps_c[i])
        i_hot  = max(range(len(log)), key=lambda i: temps_c[i])
        dT = temps_c[i_hot] - temps_c[i_cold]

        def slice_active(e):
            off = lo - e["lo_r"]
            return e["delta_ps_active"][off: off + (hi - lo + 1)]

        series_cold, series_hot = slice_active(log[i_cold]), slice_active(log[i_hot])
        diffs = [h - c for c, h in zip(series_cold, series_hot)]

        print(f"\nConfronto: voce [{i_cold}] {temps_c[i_cold]:.2f}°C  vs  "
              f"voce [{i_hot}] {temps_c[i_hot]:.2f}°C   (ΔT = {dT:+.2f}°C, "
              f"{hi - lo + 1} tap in comune)")
        if abs(dT) < 0.5:
            print("⚠️  Temperature troppo vicine (<0.5°C) per una stima affidabile — "
                  "servono calibrazioni prese in condizioni termiche più diverse.")
        else:
            mean_drift = sum(diffs) / len(diffs)
            max_drift  = max(diffs, key=abs)
            print(f"deriva media:   {mean_drift:+.3f} ps  ({mean_drift/dT:+.4f} ps/°C)")
            print(f"deriva massima: {max_drift:+.3f} ps  ({max_drift/dT:+.4f} ps/°C)")

        # grafico: tutte le curve salvate sovrapposte, colorate per temperatura
        if "show" not in dir():
            import sys, io
            sys.modules.pop("matplotlib.pyplot", None)
            import matplotlib
            matplotlib.use("Agg", force=True)
            import matplotlib.pyplot as plt
            from IPython.display import Image, display
            def show(fig):
                buf = io.BytesIO()
                fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
                plt.close(fig)
                display(Image(data=buf.getvalue()))
        else:
            import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(11, 3.5))
        tmin, tmax = min(temps_c), max(temps_c)
        for e, t in zip(log, temps_c):
            frac = 0.5 if tmax == tmin else (t - tmin) / (tmax - tmin)
            color = plt.cm.coolwarm(frac)
            ax.plot(range(e["lo_r"], e["hi_r"] + 1), e["delta_ps_active"],
                    lw=1.0, color=color, alpha=0.8, label=f"{t:.1f}°C")
        ax.set_xlabel("tap (fine)"); ax.set_ylabel("delta_ps")
        ax.set_title("Curve di calibrazione salvate, colore = temperatura PL "
                      "(blu=fredda, rosso=calda)")
        ax.legend(loc="best", fontsize=8, ncol=min(len(log), 4))
        ax.grid(alpha=.3)
        show(fig)